# Normalizacao do Relatorio de Despesas do Sistema ATUA

Este notebook le o arquivo `Relatorio_Despesas_Sistema-ATUA_{MM}.xls` (sistema ATUA, GSL Logistica) e converte para o layout do fechamento SAGI (`FECHAMENTO_ODBC_{AAAA}_{MM}.xlsx`).

Diferente do ATUA, o SAGI classifica a GSL na divisao **TRANSMOVE GSL (1.4)** com quatro filiais e dois departamentos analiticos (TRANSPORTE / ADMINISTRATIVO). Este notebook:

1. Ajuste o parametro `MES_REFERENCIA` na primeira celula de codigo (formato `MM/AAAA`). Para relatorios consolidados de varios meses (ex.: `Relatorio_Despesas_Sistema-ATUA_Jan-Fev.xls`), preencha tambem `PERIODO_ARQUIVO = "Jan-Fev"` na mesma celula
2. Le os dados da aba `base` (se nao existir, usa a primeira aba da planilha)
3. Aplica a **dinamica de tratamento de despesas** (o relatorio **bruto** tera mais linhas e soma maior que o arquivo de fechamento):
   - Exclui historicos `{225, 230, 231, 237, 238}` (Remessa, Devolucao de peca, Pamcard, Center Pecas & afins, Transferencias entre filiais). Ajuste `CD_HISTORICO_EXCLUIR` / `APLICAR_FILTRO_HISTORICO` na celula do filtro se a dinamica mudar.
   - Exclui linhas com `nm_unidade_centro_custo` contendo `CUSTO JA ALOCADO` (evita duplicidade com nota-mae; alinhado as linhas da premissa com esse texto)
4. Converte o Centro de Custo do ATUA (`cd_unidade` + `cd_centro_custo`) para o padrao SAGI
5. Converte o Plano de Contas do ATUA (`cd_historico` / `nm_historico`) para o padrao SAGI
6. Gera um Excel no layout FECHAMENTO_ODBC


In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# ── Parametros ──────────────────────────────────────────────────────────────
MES_REFERENCIA = "01/2026"   # formato MM/AAAA — base do modelo de fechamento
ATUA_SUBPASTA = "Jan"       # ex.: "Maio", "Abril", "Março"; None = raiz
ARQUIVO_ENTRADA_NOME = "Relatorio_Despesas_Sistema-ATUA_01.xls"

# Opcional: sufixo textual do arquivo de saida quando o relatorio
# nao for de um unico mes numerico (ex.: "Jan-Fev").
PERIODO_ARQUIVO = None

# Opcional: mes/ano do arquivo-modelo FECHAMENTO_ODBC a usar como referencia.
# Se None, usa o mesmo mes de MES_REFERENCIA e, se nao existir, cai para o ultimo disponivel.
MODELO_MES = "04/2026"
# ────────────────────────────────────────────────────────────────────────────

mes_num, ano = MES_REFERENCIA.split("/")
sufixo_entrada = PERIODO_ARQUIVO if PERIODO_ARQUIVO else mes_num
sufixo_saida   = PERIODO_ARQUIVO if PERIODO_ARQUIVO else f"{mes_num}-{ano}"

REFS_DIR = Path("../../02-Referencias")
ATUA_DIR = REFS_DIR / "ATUA"
ATUA_MES_DIR = (ATUA_DIR / ATUA_SUBPASTA) if ATUA_SUBPASTA else ATUA_DIR

if ARQUIVO_ENTRADA_NOME:
    ARQUIVO_ENTRADA = ATUA_MES_DIR / ARQUIVO_ENTRADA_NOME
else:
    ARQUIVO_ENTRADA = ATUA_MES_DIR / f"Relatorio_Despesas_Sistema-ATUA_{sufixo_entrada}.xls"

ARQUIVO_SAIDA = ATUA_MES_DIR / f"ATUA_despesas_fechamento_{sufixo_saida}.xlsx"

if MODELO_MES:
    _mod_mes, _mod_ano = MODELO_MES.split("/")
    ARQUIVO_MODELO_FECHAMENTO = REFS_DIR / f"FECHAMENTO_ODBC_{_mod_ano}_{_mod_mes}.xlsx"
else:
    ARQUIVO_MODELO_FECHAMENTO = REFS_DIR / f"FECHAMENTO_ODBC_{ano}_{mes_num}.xlsx"

if not ARQUIVO_MODELO_FECHAMENTO.exists():
    _candidatos = sorted(REFS_DIR.glob("FECHAMENTO_ODBC_*.xlsx"))
    if not _candidatos:
        _candidatos = sorted((REFS_DIR / "outros").glob("FECHAMENTO_ODBC_*.xlsx"))
    if not _candidatos:
        _candidatos = [
            ATUA_MES_DIR / f"ATUA_receitas_fechamento_{sufixo_saida}.xlsx",
            ATUA_MES_DIR / f"ATUA_custo_frete_fechamento_{sufixo_saida}.xlsx",
            ATUA_DIR / "Abril" / "ATUA_despesas_fechamento_04-2026.xlsx",
        ]
        _candidatos = [p for p in _candidatos if p.exists()]
    if _candidatos:
        ARQUIVO_MODELO_FECHAMENTO = _candidatos[-1] if isinstance(_candidatos[-1], Path) else _candidatos[-1]
        print(f"[AVISO] Modelo ODBC nao encontrado. Usando como referencia: {ARQUIVO_MODELO_FECHAMENTO.name}")
    else:
        raise FileNotFoundError(
            f"Nenhum modelo FECHAMENTO_ODBC ou fechamento ATUA de referencia encontrado em {REFS_DIR.resolve()}"
        )

if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(f"Arquivo ATUA nao encontrado: {ARQUIVO_ENTRADA.resolve()}")

ATUA_MES_DIR.mkdir(parents=True, exist_ok=True)

COLUNAS_INTERESSE = [
    "dt_lancamento",
    "nm_pessoa_favorecido",
    "cd_historico",
    "nm_historico",
    "nm_pessoa_filial",
    "dt_lancamento_",
    "vl_lancamento",
    "vl_lancamento_liquido",
    "ds_complemento",
    "cd_centro_custo",
    "nm_centro_custo",
    "cd_unidade",
    "nm_unidade",
    "nm_unidade_centro_custo",
    "nr_documento",
]

# Leitura robusta: alguns meses trazem linha de total antes do cabecalho.
raw_atua = pd.read_excel(
    ARQUIVO_ENTRADA,
    sheet_name=0,
    header=None,
    dtype=object,
    engine="xlrd",
    engine_kwargs={"ignore_workbook_corruption": True},
)

header_idx = None
for i in range(min(30, len(raw_atua))):
    vals = [str(v).strip().lower() for v in raw_atua.iloc[i].tolist() if pd.notna(v)]
    if "dt_lancamento_" in vals and "vl_lancamento_liquido" in vals:
        header_idx = i
        break
if header_idx is None:
    raise ValueError("Cabecalho de despesas nao encontrado (dt_lancamento_ / vl_lancamento_liquido).")

df_atua = raw_atua.iloc[header_idx + 1 :].copy()
df_atua.columns = [str(c).strip() for c in raw_atua.iloc[header_idx].tolist()]
df_atua = df_atua.reset_index(drop=True)
mask_vazia = df_atua.apply(
    lambda r: all(pd.isna(v) or str(v).strip() == "" for v in r.values), axis=1
)
df_atua = df_atua.loc[~mask_vazia].reset_index(drop=True)

import sys
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import normalizar_colunas_data

df_atua = normalizar_colunas_data(df_atua, ("dt_lancamento_", "dt_lancamento"))

# Compatibilidade entre exportacoes ATUA (nomes de colunas variam por versao/relatorio).
if "unidade_centro_custo" in df_atua.columns and "nm_unidade_centro_custo" in df_atua.columns:
    _ucc = df_atua["nm_unidade_centro_custo"].astype(str).str.strip()
    _leg = df_atua["unidade_centro_custo"].astype(str).str.strip()
    df_atua["nm_unidade_centro_custo"] = _ucc.where(
        _ucc.ne("") & _ucc.str.lower().ne("nan"), _leg
    )
    df_atua = df_atua.drop(columns=["unidade_centro_custo"])
elif "unidade_centro_custo" in df_atua.columns:
    df_atua = df_atua.rename(columns={"unidade_centro_custo": "nm_unidade_centro_custo"})

# Evita colunas duplicadas no cabecalho (pandas retorna DataFrame em df["col"]).
if df_atua.columns.duplicated().any():
    df_atua = df_atua.loc[:, ~df_atua.columns.duplicated()].copy()

def _int_str_local(v) -> str:
    if pd.isna(v):
        return ""
    try:
        f = float(v)
        if f.is_integer():
            return str(int(f))
    except (TypeError, ValueError):
        pass
    return str(v).strip()

_MAPA_UNIDADE_POR_FILIAL = {
    "GSL PRUDENTE": "9",
    "GSL DOURADOS": "16",
    "GSL MARINGA PR": "12",
    "GSL BARUERI": "26",
}
_MAPA_DEP_TRANSPORTE = {"62"}  # FRETES PAGOS

if "cd_unidade" not in df_atua.columns:
    df_atua["cd_unidade"] = df_atua.get("nm_pessoa_filial", "").map(_MAPA_UNIDADE_POR_FILIAL)
    print("[AVISO] Colunas de CC ausentes no XLS; cd_unidade inferido por nm_pessoa_filial.")
if "nm_unidade" not in df_atua.columns:
    df_atua["nm_unidade"] = df_atua.get("nm_pessoa_filial", "")
if "cd_centro_custo" not in df_atua.columns:
    df_atua["cd_centro_custo"] = df_atua.get("cd_historico", "").map(
        lambda h: "81" if _int_str_local(h) in _MAPA_DEP_TRANSPORTE else "100"
    )
    print("[AVISO] cd_centro_custo ausente; usando 81 (TRANSPORTE) para FRETES PAGOS, senao 100 (ADMIN).")
if "nm_centro_custo" not in df_atua.columns:
    df_atua["nm_centro_custo"] = df_atua["cd_centro_custo"].map(
        {"81": "TRANSPORTE", "100": "ADMINISTRATIVO/COMERCIAL", "102": "FROTA TERCEIRO PRUDENTE"}
    ).fillna("ADMINISTRATIVO/COMERCIAL")
if "nm_unidade_centro_custo" not in df_atua.columns:
    df_atua["nm_unidade_centro_custo"] = ""

_cols_faltando = [c for c in COLUNAS_INTERESSE if c not in df_atua.columns]
if _cols_faltando:
    raise KeyError(f"Colunas ausentes no relatorio de despesas: {_cols_faltando}")

df_atua = df_atua[COLUNAS_INTERESSE].copy()
print(f"Header detectado na linha: {header_idx}")

print(f"Entrada: {ARQUIVO_ENTRADA.resolve()}")
print(f"Linhas lidas: {len(df_atua)}")
print(f"Colunas usadas: {df_atua.columns.tolist()}")
df_atua.head(5)

In [ ]:
## Filtro de historicos a desconsiderar (Dinamica)
#
# Codigos excluidos conforme dinamica de tratamento do relatorio de despesas:
#   225 - Remessa
#   230 - Devolucao de peca (analisar todas)
#   231 - Pamcard (combustivel ja lancado via SAGI)
#   237 - Center Pecas / Distribuidora Automotiva / Odapel / Pellegrino
#   238 - Transferencias entre filiais
#
# Importante: o total de linhas e a soma de vl_lancamento_liquido do relatorio **bruto**
# serao **maiores** que os do arquivo de fechamento, porque estas linhas saem aqui.
# A soma para conferir com o Excel gerado e a da base **apos** este filtro (ver print abaixo).
APLICAR_FILTRO_HISTORICO = True
CD_HISTORICO_EXCLUIR = {225, 230, 231, 237, 238}

def _cd_int(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None

if APLICAR_FILTRO_HISTORICO:
    mask_excluir = df_atua["cd_historico"].apply(lambda v: _cd_int(v) in CD_HISTORICO_EXCLUIR)
    n_excluidas = int(mask_excluir.sum())

    if n_excluidas:
        soma_liquido_excluidas = pd.to_numeric(
            df_atua.loc[mask_excluir, "vl_lancamento_liquido"],
            errors="coerce",
        ).sum()
        print(f"Historicos desconsiderados: {sorted(CD_HISTORICO_EXCLUIR)}")
        print("Linhas excluidas:")
        print(
            df_atua.loc[
                mask_excluir,
                ["cd_historico", "nm_historico", "nm_pessoa_filial", "vl_lancamento", "vl_lancamento_liquido"],
            ].to_string(index=True)
        )
        print(f"\nSoma vl_lancamento_liquido nessas linhas excluidas: {soma_liquido_excluidas:,.2f}")
        df_atua = df_atua[~mask_excluir].reset_index(drop=True)
        print(f"\nLinhas restantes para processamento: {len(df_atua)}")
    else:
        print(f"Nenhuma linha com cd_historico em {sorted(CD_HISTORICO_EXCLUIR)} encontrada.")
        print(f"Linhas para processamento: {len(df_atua)}")

    soma_base_apos_historico = pd.to_numeric(df_atua["vl_lancamento_liquido"], errors="coerce").sum()
    print(
        f"Soma vl_lancamento_liquido na base apos filtro de historico: {soma_base_apos_historico:,.2f} "
        "(conferir com SUM de valor_nf/valor_pago/valor_conta no Excel de saida)."
    )
else:
    print("Filtro de historico desativado (APLICAR_FILTRO_HISTORICO=False).")
    print(f"Linhas para processamento: {len(df_atua)}")


## Filtro de linhas com "CUSTO JA ALOCADO"

Linhas cuja `nm_unidade_centro_custo` contem `CUSTO JA ALOCADO` representam custos
que ja foram alocados a outra filial/centro e nao devem ser contabilizados novamente
no fechamento SAGI (evita duplicidade). Exemplos comuns: `2 / CUSTO JA ALOCADO - NOTA MAE`,
`3 / FROTA - 2 / CUSTO JA ALOCADO - NOTA MAE`.

In [ ]:
FILTRAR_CUSTO_JA_ALOCADO = True

if FILTRAR_CUSTO_JA_ALOCADO:
    mask_alocado = df_atua["nm_unidade_centro_custo"].astype(str).str.contains(
        "CUSTO JA ALOCADO", case=False, na=False
    )
    n_alocado = int(mask_alocado.sum())

    if n_alocado:
        print(f"[EXCLUIDO] {n_alocado} linha(s) com 'CUSTO JA ALOCADO' em nm_unidade_centro_custo:")
        print(
            df_atua.loc[
                mask_alocado,
                ["cd_unidade", "nm_unidade_centro_custo", "vl_lancamento_liquido"],
            ].to_string(index=True)
        )
        df_atua = df_atua[~mask_alocado].reset_index(drop=True)
        print(f"\nLinhas restantes para processamento: {len(df_atua)}")
    else:
        print("Nenhuma linha com 'CUSTO JA ALOCADO' encontrada.")
        print(f"Linhas para processamento: {len(df_atua)}")
else:
    print("Filtro 'CUSTO JA ALOCADO' desativado (FILTRAR_CUSTO_JA_ALOCADO=False).")
    print(f"Linhas para processamento: {len(df_atua)}")

## Mapa de Centros de Custo (ATUA -> SAGI)

O ATUA classifica o CC com dois codigos: `cd_unidade` (filial) + `cd_centro_custo` (departamento).
No SAGI, a GSL e a divisao **1.4 TRANSMOVE GSL**, com 4 filiais:

| cd_unidade ATUA | Filial ATUA | Filial SAGI | Codigo SAGI nivel 3 |
|---|---|---|---|
| 8 | FROTA TERCEIRO PRUDENTE | PRESIDENTE PRUDENTE | 1.4.1 |
| 9 | PRUDENTE ADMINISTRATIVO/COMERCIAL | PRESIDENTE PRUDENTE | 1.4.1 |
| 12 | MARINGA ADMINISTRATIVO/COMERCIAL | MARINGA | 1.4.3 |
| 16 | DOURADOS ADMINISTRATIVO/COMERCIAL | DOURADOS | 1.4.2 |
| 26 | BARUERI ADMINISTRATIVO/COMERCIAL | BARUERI | 1.4.4 |

E dois departamentos analiticos:

| cd_centro_custo ATUA | Nome ATUA | Departamento SAGI | Sufixo codigo |
|---|---|---|---|
| 81 | Transporte | TRANSPORTE | .1 |
| 95 | Sucata | TRANSPORTE | .1 |
| 100 | ADMINISTRATIVO/COMERCIAL | ADMINISTRATIVO | .2 |
| 102 | FROTA TERCEIRO PRUDENTE | TRANSPORTE | .1 |

In [ ]:
def _str(v) -> str:
    if pd.isna(v):
        return ""
    return str(v).strip()

def _int_str(v) -> str:
    """Converte floats como 9.0 para '9' (o pandas as vezes le inteiros como float)."""
    if pd.isna(v):
        return ""
    try:
        f = float(v)
        if f.is_integer():
            return str(int(f))
    except (TypeError, ValueError):
        pass
    return _str(v)

print("=== Valores unicos de filial/unidade no ATUA ===")
print(df_atua[["cd_unidade", "nm_unidade"]].drop_duplicates().sort_values("cd_unidade").to_string(index=False))
print()
print("=== Valores unicos de departamento no ATUA ===")
print(df_atua[["cd_centro_custo", "nm_centro_custo"]].drop_duplicates().sort_values("cd_centro_custo").to_string(index=False))

MAPA_FILIAL = {
    "8":  {"n3_cod": "1.4.1", "n3_desc": "PRESIDENTE PRUDENTE", "filial_saida": "GSL PRUDENTE"},
    "9":  {"n3_cod": "1.4.1", "n3_desc": "PRESIDENTE PRUDENTE", "filial_saida": "GSL PRUDENTE"},
    "12": {"n3_cod": "1.4.3", "n3_desc": "MARINGA",             "filial_saida": "GSL MARINGA"},
    "16": {"n3_cod": "1.4.2", "n3_desc": "DOURADOS",            "filial_saida": "GSL DOURADOS"},
    "26": {"n3_cod": "1.4.4", "n3_desc": "BARUERI",             "filial_saida": "GSL BARUERI"},
}

MAPA_DEPARTAMENTO = {
    "81":  {"sufixo": "1", "desc": "TRANSPORTE"},
    "95":  {"sufixo": "1", "desc": "TRANSPORTE"},
    "100": {"sufixo": "2", "desc": "ADMINISTRATIVO"},
    "102": {"sufixo": "1", "desc": "TRANSPORTE"},
}

def mapear_cc(cd_unidade, cd_centro_custo) -> dict | None:
    """Retorna o codigo SAGI completo + toda a hierarquia ou None se nao for mapeavel.

    Hierarquia de 3 niveis uteis (o prefixo numerico nao conta como nivel):
      n1: 1.4            -> TRANSMOVE GSL
      n2: 1.4.X          -> filial (ex: 1.4.1 PRESIDENTE PRUDENTE)
      n3: 1.4.X.Y        -> setor  (ex: 1.4.1.1 TRANSPORTE)
      n4: igual a n3     -> padrao do modelo FECHAMENTO_ODBC
    """
    uni = _int_str(cd_unidade)
    dep = _int_str(cd_centro_custo)
    filial = MAPA_FILIAL.get(uni)
    depto = MAPA_DEPARTAMENTO.get(dep)
    if not filial or not depto:
        return None
    cod_setor = f"{filial['n3_cod']}.{depto['sufixo']}"
    return {
        "n1_cod": "1.4",
        "n1_desc": "TRANSMOVE GSL",
        "n2_cod": filial["n3_cod"],
        "n2_desc": filial["n3_desc"],
        "n3_cod": cod_setor,
        "n3_desc": depto["desc"],
        "n4_cod": cod_setor,
        "n4_desc": depto["desc"],
        "filial_saida": filial["filial_saida"],
        "segmento": "TRANSMOVE GSL",
    }

ccs_nao_mapeados_cc = []
for _, r in df_atua[["cd_unidade", "cd_centro_custo", "nm_unidade", "nm_centro_custo"]].drop_duplicates().iterrows():
    if mapear_cc(r["cd_unidade"], r["cd_centro_custo"]) is None:
        ccs_nao_mapeados_cc.append(
            (_int_str(r["cd_unidade"]), _str(r["nm_unidade"]), _int_str(r["cd_centro_custo"]), _str(r["nm_centro_custo"]))
        )

if ccs_nao_mapeados_cc:
    print("\n[ALERTA] Centros de Custo nao mapeados:")
    for item in ccs_nao_mapeados_cc:
        print(f"  cd_unidade={item[0]} ({item[1]}) | cd_centro_custo={item[2]} ({item[3]})")
else:
    print("\nTodos os centros de custo foram mapeados com sucesso.")

## Mapa de Plano de Contas (ATUA -> SAGI)

O ATUA usa `cd_historico` (codigo) e `nm_historico` (descricao) como plano de contas. Esta celula mapeia cada historico ATUA para o codigo oficial do SAGI (`02-Referencias/Plano de Contas.pdf`):

| cd_historico ATUA | nm_historico ATUA | Codigo SAGI | Descricao SAGI |
|---|---|---|---|
| 15 | ENERGIA ELETRICA | 7.5.2 | ENERGIA ELETRICA |
| 16 | ALUGUEL E CONDOMINIOS | 7.5.31 | ALUGUEL ADMINISTRATIVO |
| 17 | SEGURO DE CARGAS | 6.6.4 | SEGURO DE CARGAS |
| 23 | TARIFAS BANCARIAS | 7.5.22 | DESPESAS BANCARIAS |
| 26 | ASSESSORIAS E TELECONSULTAS | 7.5.9 | CONSULTORIA |
| 46 | IMPOSTOS E TAXAS DIVERSAS | 7.5.17 | TAXAS |
| 62 | FRETES PAGOS | 6.6.1 | FRETE DE TERCEIROS |
| 69 | HONORARIOS CONTABEIS | 7.5.7 | HONORARIOS CONTABEIS |
| 95 | ICMS | 7.4.12 | ICMS |
| 209 | PESSOAL - INSS PATRONAL | 7.3.3 | INSS |
| 229 | PESSOAL - PRO LABORE | 7.3.12 | PRO LABORE |
| 231 | DIESEL - PAMCARD | 7.1.4 | COMBUSTIVEL - DIESEL (POSTO) |

In [ ]:
print("=== Valores unicos de Plano de Contas no ATUA ===")
print(df_atua[["cd_historico", "nm_historico"]].drop_duplicates().sort_values("cd_historico").to_string(index=False))

MAPA_PLANO_CONTAS = {
    "15":  {"cod": "7.5.2",  "desc": "ENERGIA ELETRICA"},
    "16":  {"cod": "7.5.31", "desc": "ALUGUEL ADMINISTRATIVO"},
    "17":  {"cod": "6.6.4",  "desc": "SEGURO DE CARGAS"},
    "23":  {"cod": "7.5.22", "desc": "DESPESAS BANCARIAS"},
    "26":  {"cod": "7.5.9",  "desc": "CONSULTORIA"},
    "31":  {"cod": "7.1.9",  "desc": "MULTAS DE TRANSITO"},
    "46":  {"cod": "7.5.17", "desc": "TAXAS"},
    "62":  {"cod": "6.6.1",  "desc": "FRETE DE TERCEIROS"},
    "69":  {"cod": "7.5.7",  "desc": "HONORARIOS CONTABEIS"},
    "95":  {"cod": "7.4.12", "desc": "ICMS"},
    "209": {"cod": "7.3.3",  "desc": "INSS"},
    "229": {"cod": "7.3.12", "desc": "PRO LABORE"},
    "231": {"cod": "7.1.4",  "desc": "COMBUSTIVEL - DIESEL (POSTO)"},
}

def mapear_plano_contas(cd_historico) -> dict | None:
    key = _int_str(cd_historico)
    return MAPA_PLANO_CONTAS.get(key)

_CREDORES_FRETES_PAGOS_NAO_FRETE = (
    "NSTECH",
    "IPC INSTITUICAO DE PAGAMENTO",
)

def _ajustar_plano_contas_despesa(row, pc_map: dict | None) -> dict | None:
    """Historico 62 com credor de instituicao de pagamento -> taxa, nao frete de terceiros."""
    if pc_map is None or _int_str(row.get("cd_historico")) != "62":
        return pc_map
    fav = _str(row.get("nm_pessoa_favorecido", "")).upper()
    if any(token in fav for token in _CREDORES_FRETES_PAGOS_NAO_FRETE):
        return {"cod": "7.5.17", "desc": "TAXAS"}
    return pc_map

historicos_nao_mapeados = []
for _, r in df_atua[["cd_historico", "nm_historico"]].drop_duplicates().iterrows():
    if mapear_plano_contas(r["cd_historico"]) is None:
        historicos_nao_mapeados.append((_int_str(r["cd_historico"]), _str(r["nm_historico"])))

if historicos_nao_mapeados:
    print("\n[ALERTA] Planos de Contas nao mapeados:")
    for cod, desc in historicos_nao_mapeados:
        print(f"  cd_historico={cod} ({desc})")
else:
    print("\nTodos os planos de contas foram mapeados com sucesso.")

## Conversao para layout FECHAMENTO_ODBC

Regras de mapeamento aplicadas (conforme dinamica de despesas):

- `nm_pessoa_filial` -> `filial`
- `dt_lancamento_` -> `data_nf` (data do lancamento financeiro)
- `data_pagamento` -> vazio (nao mapeado pela dinamica)
- `vl_lancamento_liquido` -> `valor_nf`, `valor_pago`, `valor_conta` (gravados como **numero** no Excel para o SUM coincidir com a base apos filtros; nao usar texto tipo `1.234,56` nestas colunas)
- `nm_pessoa_favorecido` -> `credor_forn_cli_func`
- `nm_centro_custo` -> `observacao`
- `nm_historico` -> `Dados auxiliares`
- `cd_unidade` + `cd_centro_custo` -> `n1_cod_centro_custo` ate `n4_cod_centro_custo` + descricoes hierarquicas
- `cd_historico` -> `cod_conta` + `conta` + `cod_conta-descr`
- `nr_documento` -> `titulo`
- `Origem` = "Saidas (Aplicacoes)" (fixo — todo o relatorio ATUA de despesas e saida)
- `Sistema` = "ATUA"


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import parse_valor_fechamento, parse_data_fechamento, gravar_fechamento_excel

def _to_float(v):
    return parse_valor_fechamento(v)



modelo_cols = pd.read_excel(ARQUIVO_MODELO_FECHAMENTO, nrows=0).columns.tolist()
print("Colunas do modelo FECHAMENTO:", modelo_cols)

linhas_saida = []
alertas_linhas_cc = []
alertas_linhas_pc = []

for i, row in df_atua.iterrows():
    cc_map = mapear_cc(row["cd_unidade"], row["cd_centro_custo"])
    pc_map = _ajustar_plano_contas_despesa(row, mapear_plano_contas(row["cd_historico"]))

    # Regra de negocio: MULTA - OUTRAS (31) e MULTAS DE TRANSITO -> setor TRANSPORTE.
    # O cd_centro_custo de origem pode ser 100 (ADMINISTRATIVO), mas multas de transito
    # pertencem ao setor de transporte; sobrescrevemos n3 e n4 para o sufixo ".1".
    if cc_map is not None and _int_str(row["cd_historico"]) == "31":
        cod_setor = f"{cc_map['n2_cod']}.1"
        cc_map = {
            **cc_map,
            "n3_cod": cod_setor,
            "n3_desc": "TRANSPORTE",
            "n4_cod": cod_setor,
            "n4_desc": "TRANSPORTE",
        }

    if cc_map is None:
        alertas_linhas_cc.append({
            "idx": i,
            "cd_unidade": _int_str(row["cd_unidade"]),
            "nm_unidade": _str(row["nm_unidade"]),
            "cd_centro_custo": _int_str(row["cd_centro_custo"]),
            "nm_centro_custo": _str(row["nm_centro_custo"]),
        })
    if pc_map is None:
        alertas_linhas_pc.append({
            "idx": i,
            "cd_historico": _int_str(row["cd_historico"]),
            "nm_historico": _str(row["nm_historico"]),
        })

    valor = _to_float(row["vl_lancamento_liquido"])

    nova = {c: pd.NA for c in modelo_cols}
    nova["id"] = i + 1

    if cc_map is not None:
        nova["Segmento"] = cc_map["segmento"]
        nova["n1_cod_centro_custo"] = cc_map["n1_cod"]
        nova["n1_centro_custo"] = cc_map["n1_desc"]
        nova["n1_CC"] = f"{cc_map['n1_cod']} {cc_map['n1_desc']}"
        nova["n2_cod_centro_custo"] = cc_map["n2_cod"]
        nova["n2_centro_custo"] = cc_map["n2_desc"]
        nova["n2_CC"] = f"{cc_map['n2_cod']} {cc_map['n2_desc']}"
        nova["n3_cod_centro_custo"] = cc_map["n3_cod"]
        nova["n3_centro_custo"] = cc_map["n3_desc"]
        nova["n3_CC"] = f"{cc_map['n3_cod']} {cc_map['n3_desc']}"
        nova["n4_cod_centro_custo"] = cc_map["n4_cod"]
        nova["n4_centro_custo"] = cc_map["n4_desc"]
        nova["n4_CC"] = f"{cc_map['n4_cod']} {cc_map['n4_desc']}"
        nova["filial"] = cc_map["filial_saida"]
    else:
        nova["filial"] = _str(row["nm_pessoa_filial"])

    if pc_map is not None:
        nova["cod_conta"] = pc_map["cod"]
        nova["conta"] = pc_map["desc"]
        nova["cod_conta-descr"] = f"{pc_map['cod']} {pc_map['desc']}"

    nova["titulo"] = _str(row["nr_documento"])
    # Gravar valores como numero no Excel: strings no formato brasileiro (ex.: "1.234,56")
    # fazem o SUM do Excel interpretar errado e inflar o total.
    if pd.isna(valor):
        nova["valor_nf"] = pd.NA
        nova["valor_pago"] = pd.NA
        nova["valor_conta"] = pd.NA
    else:
        v = float(valor)
        nova["valor_nf"] = v
        nova["valor_pago"] = -abs(v)
        nova["valor_conta"] = -abs(v)
    nova["observacao"] = _str(row["nm_centro_custo"])
    nova["data_nf"] = parse_data_fechamento(row["dt_lancamento_"])
    nova["data_pagamento"] = pd.NaT
    nova["credor_forn_cli_func"] = _str(row["nm_pessoa_favorecido"])
    nova["Dados auxiliares"] = _str(row["nm_historico"])
    nova["Origem"] = "Saidas (Aplicacoes)"
    nova["Sistema"] = "ATUA"

    linhas_saida.append(nova)

fechamento_df = pd.DataFrame(linhas_saida, columns=modelo_cols)

_soma_liquido_base = pd.to_numeric(df_atua["vl_lancamento_liquido"], errors="coerce").sum()
_soma_valor_saida = pd.to_numeric(fechamento_df["valor_conta"], errors="coerce").abs().sum()
print(
    f"\nConferencia de valores: soma vl_lancamento_liquido (base apos filtros) = {_soma_liquido_base:,.2f}; "
    f"soma valor_conta (saida) = {_soma_valor_saida:,.2f}"
)

print(f"\nLinhas base ATUA: {len(df_atua)}")
print(f"Linhas geradas: {len(fechamento_df)}")
if len(fechamento_df) != len(df_atua):
    raise ValueError(
        f"Divergencia de linhas: base={len(df_atua)} vs fechamento={len(fechamento_df)}."
    )
print(f"Linhas com CC nao mapeado: {len(alertas_linhas_cc)}")
print(f"Linhas com PC nao mapeado: {len(alertas_linhas_pc)}")
fechamento_df.head(10)


## Salvamento e relatorio

Gera o Excel final em `02-Referencias/ATUA/ATUA_despesas_fechamento_{MM}-{AAAA}.xlsx` e imprime o relatorio de itens nao mapeados para revisao manual.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import parse_valor_fechamento, parse_data_fechamento, gravar_fechamento_excel

import datetime

SHEET_NAME = "ATUA_despesas_fechamento"

arquivo_saida_exec = ARQUIVO_SAIDA
try:
    gravar_fechamento_excel(fechamento_df, arquivo_saida_exec, sheet_name=SHEET_NAME)
    print(f"Arquivo gerado: {arquivo_saida_exec.resolve()}")
except PermissionError:
    ts = datetime.datetime.now().strftime("%H%M%S")
    arquivo_saida_exec = ARQUIVO_SAIDA.with_stem(f"{ARQUIVO_SAIDA.stem}_{ts}")
    gravar_fechamento_excel(fechamento_df, arquivo_saida_exec, sheet_name=SHEET_NAME)
    print(
        f"[AVISO] Arquivo principal em uso ({ARQUIVO_SAIDA.name}). "
        f"Salvo como: {arquivo_saida_exec.resolve()}"
    )

print(f"Linhas gravadas: {len(fechamento_df)}")

if ccs_nao_mapeados_cc:
    print("\n[REVISAO MANUAL] Centros de Custo nao mapeados:")
    for item in ccs_nao_mapeados_cc:
        print(f"  cd_unidade={item[0]} ({item[1]}) | cd_centro_custo={item[2]} ({item[3]})")

if historicos_nao_mapeados:
    print("\n[REVISAO MANUAL] Historicos (Plano de Contas) nao mapeados:")
    for cod, desc in historicos_nao_mapeados:
        print(f"  cd_historico={cod} ({desc})")

if not ccs_nao_mapeados_cc and not historicos_nao_mapeados:
    print("\nNenhum item pendente. Todos os centros de custo e planos de conta foram mapeados.")